# Hybrid ALNS — Hyperparameter Tuning

**Goal:** Minimise mean absolute gap `(bins − lower_bound)` across 5 benchmark datasets (710 instances) by tuning the 9 metaheuristic knobs of the Hybrid ALNS solver.

## Pipeline Overview

| Step | What | Output file |
|---|---|---|
| **0. Baseline** | Evaluate default config (reference point) | `baseline_defaults.json` |
| **1. Isolation — SA Thermal** | Grid-search SA params with uniform-random bandit | `isolation_step1_sa_thermal.json` |
| **2. Isolation — Destruction** | Grid-search k/no_improve with uniform-random bandit | `isolation_step2_destruction.json` |
| **3. Isolation — Bandit** | Grid-search bandit params with bandit ON | `isolation_step3_bandit.json` |
| **4. Isolation merged** | Best from each group merged into one config | `isolation_merged.json` |
| **5. Stage 1 (coarse)** | Wide-range Bayesian search (2 000 iter/inst, 100 trials) | `alns_stage1_coarse_study.json` |
| **6. Stage 2 (fine)** | Narrow windows around Stage-1 best (5 000 iter/inst, 50 trials) | `alns_stage2_fine_study.json` |
| **7. Final** | Best config re-evaluated on full iterations | `best_config_evaluation.json` |

**Instance bank:** 100 instances stratified by item-count buckets to cover small, medium, and large problems without overfitting to any single dataset.

**To run the full pipeline:**
```bash
cd bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/parameter_tuning/
python tune_with_optuna.py --stage1-trials 100 --stage2-trials 50
```

**To skip the isolation warm-up (go straight to Optuna):**
```bash
python tune_with_optuna.py --no-isolation --stage1-trials 100 --stage2-trials 50
```

**To only evaluate the baseline (no tuning):**
```bash
python tune_with_optuna.py --baseline-only
```

In [ ]:
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import numpy as np
import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
    plot_contour,
)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore", category=DeprecationWarning)

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

In [ ]:
OUTPUT_DIR = Path("tuning_results")

---
## 1. Parameter Taxonomy

The solver exposes 15 parameters. We divide them into **9 tunable** (the metaheuristic control surface) and **6 structural** (safety floors / design constants held at defaults).

In [ ]:
DEFAULT_PARAMS: dict[str, float | int] = {
    "initial_temperature": 1.0 / math.log(2.0),   # ≈ 1.4427
    "alpha_cool": 0.9995,
    "k_min_frac": 0.05,
    "k_max_frac": 0.25,
    "bandit_alpha": 0.3,
    "warmup_calls": 300,
    "no_improve_frac": 0.05,
    "reheat_soft_mult": 0.35,
    "reheat_hard_mult": 0.20,
}

STRUCTURAL_DEFAULTS: dict[str, float | int] = {
    "min_no_improve_limit": 250,
    "temp_precision_floor": 1e-12,
    "reheat_check_interval_divisor": 4.0,
    "reheat_check_min_interval": 50,
    "hard_restart_min_limit": 100,
    "patience_shrink_factor": 2.0 / 3.0,
}

In [ ]:
import pandas as pd

tunable_rows = [
    ("initial_temperature", "1.4427", "(0.1, 10.0) log", "Starting SA temperature; controls early acceptance of worse solutions"),
    ("alpha_cool", "0.9995", "(0.990, 0.9999)", "Cooling factor per iteration; slower cooling = more exploration"),
    ("k_min_frac", "0.05", "(0.01, 0.20)", "Minimum destruction radius as fraction of item count"),
    ("k_max_frac", "0.25", "(0.10, 0.50)", "Maximum destruction radius; grows adaptively with stagnation"),
    ("bandit_alpha", "0.3", "(0.01, 2.0) log", "LinUCB exploration bonus; higher = more exploration between arms"),
    ("warmup_calls", "300", "(0, 1000)", "Thompson Sampling warm-up iterations before switching to LinUCB"),
    ("no_improve_frac", "0.05", "(0.01, 0.20)", "Stagnation budget as fraction of max_iterations; triggers hard restart"),
    ("reheat_soft_mult", "0.35", "(0.10, 0.90)", "Temperature multiplier for soft reheats (periodic mild rewarm)"),
    ("reheat_hard_mult", "0.20", "(0.05, 0.80)", "Temperature multiplier for hard restarts (return to best + reheat)"),
]

structural_rows = [
    ("min_no_improve_limit", "250", "Safety floor; real lever is `no_improve_frac × max_iter`"),
    ("temp_precision_floor", "1e-12", "Division guard for T/T0 context feature"),
    ("reheat_check_interval_divisor", "4.0", "Tied to `no_improve_frac`; controls soft-reheat frequency"),
    ("reheat_check_min_interval", "50", "Prevents excessive reheats on short runs"),
    ("hard_restart_min_limit", "100", "Prevents restart thrashing after shrinkage"),
    ("patience_shrink_factor", "2/3", "Shrink factor for no_improve_limit on hard restart; changed from 0.5"),
]

df_tunable = pd.DataFrame(tunable_rows, columns=["Parameter", "Default", "Stage-1 Range", "Role"])
df_structural = pd.DataFrame(structural_rows, columns=["Parameter", "Default", "Why Held Fixed"])

display(df_tunable.style
    .set_caption("Tunable Parameters (9)")
    .set_table_styles([
        {"selector": "caption", "props": "font-size: 1.2em; font-weight: bold; margin-bottom: 6px;"},
        {"selector": "th", "props": "text-align: left;"},
    ])
)
display(df_structural.style
    .set_caption("Structural Parameters (6 — held fixed)")
    .set_table_styles([
        {"selector": "caption", "props": "font-size: 1.2em; font-weight: bold; margin-bottom: 6px;"},
        {"selector": "th", "props": "text-align: left;"},
    ])
)

---
## 2. Methodology: 3-Stage Tuning Pipeline

### 2a. Isolation Steps (group-wise grid search)

Before Optuna, we run three small grid searches — one for each parameter group —
with the bandit **forced to uniform random** (``force_uniform_random=True``).
This prevents the bandit's adaptive behaviour from confounding the measurement,
giving us an unbiased estimate of each group's isolated effect.

| Group | Params | Grid size | Uniform? |
|---|---|---|---|
| SA Thermal | `initial_temperature`, `alpha_cool`, `reheat_soft_mult`, `reheat_hard_mult` | 5×5×5×5 = 625 | Yes |
| Destruction | `k_min_frac`, `k_max_frac`, `no_improve_frac` | 5×5×5 = 125 | Yes |
| Bandit | `bandit_alpha`, `warmup_calls` | 6×6 = 36 | **No** (needs bandit ON) |

The best configs from each group are **merged** into a *pre-tuned anchor* that seeds
the Optuna stages. All intermediate results are saved as ``isolation_step*.json``.

### 2b. 2-Stage Bayesian Optimisation (Optuna)

| Aspect | Stage 1 (Coarse) | Stage 2 (Fine) |
|---|---|---|
| **Purpose** | Explore wide space, find the promising basin | Exploit the basin with high-fidelity evaluations |
| **Trials** | 100 | 50 |
| **Iterations / instance** | 2 000 | 5 000 |
| **Parameter ranges** | Broad (see table above) | ±15–25% around Stage-1 best |
| **Budget (~single core)** | ~100 × 100 × 5 s ≈ 14 h | ~50 × 100 × 25 s ≈ 35 h |
| **Sampler** | Optuna `TPESampler` (default) | Same |

**Instance bank** — 100 instances from all 5 datasets, stratified by item count:
| Dataset | Instances | Item-count range |
|---|---|---|
| Falkenauer-T | 5 | 60 |
| Falkenauer-U | 40 | 120 – 1 000 |
| Scholl-1 | 10 | 50 |
| Scholl-2 | 35 | 50 – 500 |
| Scholl-3 | 10 | 200 |

**Objective:** minimise **mean absolute gap** `(bins − lower_bound)` over all bank instances.
We use absolute (not relative) gap to avoid divide-by-near-zero when LB is small.

---
## 3. Output Files

All results are saved to ``tuning_results/``.

| File | Contents |
|---|---|
| ``baseline_defaults.json`` | Default config evaluation (reference point) |
| ``isolation_step1_sa_thermal.json`` | Best SA params (uniform bandit) |
| ``isolation_step2_destruction.json`` | Best destruction params (uniform bandit) |
| ``isolation_step3_bandit.json`` | Best bandit params (bandit ON) |
| ``isolation_merged.json`` | Merged best from all isolation steps |
| ``alns_stage1_coarse_study.json`` | Full Optuna study (Stage 1) |
| ``alns_stage2_fine_study.json`` | Full Optuna study (Stage 2) |
| ``best_config_evaluation.json`` | Final best config on full iterations |

Each JSON contains ``parameters`` (what was evaluated), ``metrics`` (gap, time, pulls),
and (for Optuna studies) ``search_ranges`` passed to the sampler.
Isolation logs also include ``changes_from_baseline`` — only the params that moved.

In [ ]:
def load_json(path: Path) -> dict | None:
    if not path.exists():
        print(f"[MISSING] {path.name} — run the tuning script first.")
        return None
    with open(path) as f:
        return json.load(f)

baseline_raw     = load_json(OUTPUT_DIR / "baseline_defaults.json")
iso_sa_raw       = load_json(OUTPUT_DIR / "isolation_step1_sa_thermal.json")
iso_dest_raw     = load_json(OUTPUT_DIR / "isolation_step2_destruction.json")
iso_bandit_raw   = load_json(OUTPUT_DIR / "isolation_step3_bandit.json")
iso_merged_raw   = load_json(OUTPUT_DIR / "isolation_merged.json")
stage1_raw       = load_json(OUTPUT_DIR / "alns_stage1_coarse_study.json")
stage2_raw       = load_json(OUTPUT_DIR / "alns_stage2_fine_study.json")
best_raw         = load_json(OUTPUT_DIR / "best_config_evaluation.json")

data_loaded = all(x is not None for x in [baseline_raw, stage1_raw, stage2_raw, best_raw])
if data_loaded:
    print("All result files loaded successfully.")
else:
    print("Run `python tune_with_optuna.py` to generate results.")

---
## 4. Stage 1 — Coarse Search

100 trials over wide ranges. Enqueued default config as trial 0.

In [ ]:
if stage1_raw:
    print(f"Trials completed:  {stage1_raw['n_trials']}")
    print(f"Best mean gap:     {stage1_raw['best_value']:.4f}")
    print(f"Best parameters:")
    for k, v in stage1_raw["best_params"].items():
        print(f"    {k:25s} = {v}")
    
    # Reconstruct study object for plotting
    stage1_study = optuna.create_study(direction="minimize", study_name="alns_stage1_coarse")
    for t in stage1_raw["trials"]:
        stage1_study.add_trial(
            optuna.trial.create_trial(
                params=t["params"],
                values=[t["value"]],
                state=optuna.trial.TrialState.COMPLETE,
            )
        )

### 4.1 Convergence

The optimisation history shows how the best-found gap improved over trials.

In [ ]:
if stage1_raw:
    fig = plot_optimization_history(stage1_study)
    fig.update_layout(width=800, height=450, title="Stage 1 — Optimisation History")
    fig.show()

### 4.2 Parameter Importance

Which knobs matter most? Optuna's hyperparameter importance (fANOVA / tree-based) ranks them.

In [ ]:
if stage1_raw:
    fig = plot_param_importances(stage1_study)
    fig.update_layout(
        width=700, height=450,
        title="Stage 1 — Parameter Importance",
    )
    fig.show()

### 4.3 Parallel Coordinates

Each line is one trial. Colour encodes the objective value. This reveals correlations and high-performing regions.

In [ ]:
if stage1_raw and len(stage1_raw["trials"]) >= 5:
    fig = plot_parallel_coordinate(stage1_study)
    fig.update_layout(width=900, height=500, title="Stage 1 — Parallel Coordinates")
    fig.show()

### 4.4 Slice Plot

One-dimensional projections — each dot is a trial, orange line is the rolling best.

In [ ]:
if stage1_raw:
    fig = plot_slice(stage1_study)
    fig.update_layout(width=900, height=500, title="Stage 1 — Slice Plot")
    fig.show()

---
## 5. Stage 2 — Fine Search

50 trials at 5 000 ALNS iterations, anchored ±15–25% around Stage 1's best.

In [ ]:
if stage2_raw:
    print(f"Trials completed:  {stage2_raw['n_trials']}")
    print(f"Best mean gap:     {stage2_raw['best_value']:.4f}")
    print(f"Best parameters:")
    for k, v in stage2_raw["best_params"].items():
        print(f"    {k:25s} = {v}")
    
    stage2_study = optuna.create_study(direction="minimize", study_name="alns_stage2_fine")
    for t in stage2_raw["trials"]:
        stage2_study.add_trial(
            optuna.trial.create_trial(
                params=t["params"],
                values=[t["value"]],
                state=optuna.trial.TrialState.COMPLETE,
            )
        )

In [ ]:
if stage2_raw:
    fig = plot_optimization_history(stage2_study)
    fig.update_layout(width=800, height=450, title="Stage 2 — Optimisation History (Fine)")
    fig.show()

if stage2_raw:
    fig = plot_param_importances(stage2_study)
    fig.update_layout(width=700, height=450, title="Stage 2 — Parameter Importance")
    fig.show()

---
## 6. Head-to-Head: Baseline vs Best

We compare the default configuration (trial 0, 5 000 iterations) with the best configuration found by Stage 2.

In [ ]:
if data_loaded:
    b = baseline_raw["metrics"]
    r = best_raw["metrics"]

    rows = []
    for k in sorted(set(list(b["per_dataset_gap"].keys()) + list(r["per_dataset_gap"].keys()))):
        rows.append({
            "Dataset": k,
            "Baseline gap": round(b["per_dataset_gap"].get(k, 0), 4),
            "Best gap": round(r["per_dataset_gap"].get(k, 0), 4),
        })
    rows.append({
        "Dataset": "**Overall**",
        "Baseline gap": round(b["mean_gap"], 4),
        "Best gap": round(r["mean_gap"], 4),
    })
    df_compare = pd.DataFrame(rows)
    df_compare["Improvement"] = df_compare["Baseline gap"] - df_compare["Best gap"]
    df_compare["% change"] = (df_compare["Improvement"] / df_compare["Baseline gap"].replace(0, np.nan) * 100).round(1).map(
        lambda x: f"{x:+.1f}%" if pd.notna(x) else "—"
    )

    display(df_compare.style
        .set_caption("Per-Dataset Gap Comparison (5 000 iterations)")
        .set_table_styles([
            {"selector": "caption", "props": "font-size: 1.2em; font-weight: bold; margin-bottom: 6px;"},
            {"selector": "th", "props": "text-align: left;"},
        ])
        .format({"Baseline gap": "{:.4f}", "Best gap": "{:.4f}", "Improvement": "{:.4f}"})
    )

In [ ]:
if data_loaded:
    # Side-by-side bar chart
    labels = list(df_compare["Dataset"][:-1])  # exclude Overall
    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(x - width/2, df_compare["Baseline gap"][:-1], width, label="Baseline", color="#4C72B0", edgecolor="white")
    ax.bar(x + width/2, df_compare["Best gap"][:-1], width, label="Best (tuned)", color="#DD8452", edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel("Mean absolute gap (bins − LB)")
    ax.set_title("Baseline vs Best — Per-Dataset Gap")
    ax.legend(frameon=True, fancybox=False)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()
    plt.show()

    # Overall
    fig, ax = plt.subplots(figsize=(5, 4))
    cats = ["Baseline", "Best"]
    vals = [df_compare["Baseline gap"].iloc[-1], df_compare["Best gap"].iloc[-1]]
    bars = ax.bar(cats, vals, color=["#4C72B0", "#DD8452"], edgecolor="white", width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{v:.4f}", ha="center", va="bottom", fontweight="bold")
    ax.set_ylabel("Mean absolute gap (bins − LB)")
    ax.set_title("Overall Mean Gap")
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=False))
    fig.tight_layout()
    plt.show()

---
## 7. Arm-Pull Analysis

How often did each destroy operator get selected? This reveals whether the bandit converged to a preference or remained exploratory.

In [ ]:
if data_loaded:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

    for ax, (label, data) in zip(axes, [
        ("Baseline", baseline_raw["metrics"]["arm_pulls"]),
        ("Best", best_raw["metrics"]["arm_pulls"]),
    ]):
        total = sum(data)
        colors = ["#4C72B0", "#DD8452", "#55A868"]
        wedges, texts, autotexts = ax.pie(
            data, labels=["Random", "Worst", "Related"],
            autopct="%1.0f%%", colors=colors, startangle=90,
            wedgeprops=dict(edgecolor="white", linewidth=1.5),
        )
        ax.set_title(f"{label} (total: {total:,})")

    fig.suptitle("Destroy-Operator Arm Pulls", fontsize=13, y=1.02)
    fig.tight_layout()
    plt.show()

---
## 8. Timing Summary

Wall-clock cost of each evaluation.

In [ ]:
if data_loaded:
    timing_rows = [
        ("Baseline", baseline_raw["metrics"]["mean_time_s"], baseline_raw["metrics"]["total_time_s"]),
        ("Best (tuned)", best_raw["metrics"]["mean_time_s"], best_raw["metrics"]["total_time_s"]),
    ]
    if stage1_raw:
        timing_rows.append(("Stage 1 (coarse)", None, stage1_raw["n_trials"] * baseline_raw["metrics"]["total_time_s"]))
    if stage2_raw:
        timing_rows.append(("Stage 2 (fine)", None, stage2_raw["n_trials"] * best_raw["metrics"]["total_time_s"]))

    df_time = pd.DataFrame(timing_rows, columns=["Run", "Mean / instance (s)", "Total (s)"])
    display(df_time.style
        .set_caption("Timing")
        .set_table_styles([
            {"selector": "caption", "props": "font-size: 1.2em; font-weight: bold; margin-bottom: 6px;"},
            {"selector": "th", "props": "text-align: left;"},
        ])
        .format({"Mean / instance (s)": "{:.2f}", "Total (s)": "{:.0f}"})
    )

---
## 9. Parameter Evolution (Every Stage)

Track how each parameter changed — or stayed the same — across every stage of the pipeline.

In [ ]:
if stage2_raw:
    # Build evolution table
    stages = {"default\n(initial)": dict(DEFAULT_PARAMS)}
    if iso_merged_raw:
        stages["after\nisolation"] = {**DEFAULT_PARAMS, **iso_merged_raw["parameters"]}
    if stage1_raw:
        stages["after\nStage 1"] = {**DEFAULT_PARAMS, **stage1_raw["best_params"]}
    stages["after\nStage 2\n(final)"] = {**DEFAULT_PARAMS, **stage2_raw["best_params"]}

    cols = ["Parameter"] + list(stages.keys())
    df_evo = pd.DataFrame(index=[p for p in DEFAULT_PARAMS])
    for label, cfg in stages.items():
        row = []
        for p in df_evo.index:
            v = cfg.get(p, "—")
            if isinstance(v, float):
                row.append(f"{v:.6g}")
            else:
                row.append(str(v))
        df_evo[label.replace(chr(10), " ")] = row

    display(df_evo.style
        .set_caption("Parameter Evolution Through the Pipeline")
        .set_table_styles([
            {"selector": "caption", "props": "font-size: 1.2em; font-weight: bold; margin-bottom: 6px;"},
            {"selector": "th", "props": "text-align: left;"},
        ])
    )

    # Full merged best (including structural)
    print("\n" + "=" * 50)
    print("  Best configuration (full, with structural defaults)")
    print("=" * 50)
    best = {**DEFAULT_PARAMS, **STRUCTURAL_DEFAULTS, **stage2_raw["best_params"]}
    for k, v in best.items():
        print(f"    {k:30s} = {v}")

---
## 10. Summary

- **3-stage tuning pipeline**: Isolation grid search → Stage 1 coarse → Stage 2 fine
- **Isolation steps** use ``force_uniform_random=True`` (uniform bandit) to measure each
  parameter group without bandit confounding; saves intermediate ``isolation_step*.json``
- **Stage 1** explored wide ranges (*~100 trials, 2000 iter/inst*)
- **Stage 2** refined around the best basin (*~50 trials, 5000 iter/inst*)
- **Instance bank** of 100 instances stratified by item count across all 5 datasets
- **6 structural parameters** held fixed at solver defaults (documented above)
- **9 tunable parameters** searched; each step's config is logged to JSON with full search ranges
- **CLI flags**: ``--stage1-trials``, ``--stage2-trials``, ``--no-isolation``, ``--baseline-only``, ``--max-iter``, ``--seed``, ``--output-dir``
- **Wait time**: see timing table above